In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_style('whitegrid')
sns.set_palette('Set2')

This cell imports all necessary libraries for data manipulation, visualization, and machine learning. It includes pandas, numpy, matplotlib, seaborn, scikit-learn modules for data loading, model selection, preprocessing, classification, and evaluation metrics. It also sets up seaborn plotting styles.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_style('whitegrid')
sns.set_palette('Set2')

This cell loads the Iris dataset, renames columns for clarity, and performs initial data exploration. It displays the dataset shape, the first 5 records, and the distribution of species. A pairplot is generated to visualize relationships between all feature pairs, color-coded by species, to understand class separability.

In [ ]:
iris_raw = load_iris(as_frame=True)
iris_df  = iris_raw.frame.copy()
iris_df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'target']
iris_df['species'] = iris_df['target'].map(dict(enumerate(iris_raw.target_names)))

print(f"Dataset shape : {iris_df.shape}")
print(f"\nFirst 5 records:")
display(iris_df.head())
print(f"\nClass distribution:")
display(iris_df['species'].value_counts())

# Pairplot maps every feature pair against each other;
# hue='species' reveals whether classes form separable clusters,
# guiding which features are most useful before any modelling
pairplot_fig = sns.pairplot(
    iris_df.drop(columns='target'),
    hue='species',
    palette='Set2',
    corner=True
)
pairplot_fig.figure.suptitle('Pairwise Feature Separability — Iris Dataset', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

This cell preprocesses the data for machine learning. It separates features (X) from the target variable (y), then splits the data into training and testing sets using `train_test_split` with stratification. Finally, a `StandardScaler` is used to scale the features, fitting only on the training data to prevent data leakage, and transforming both training and testing sets.

In [ ]:
feature_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = iris_df[feature_cols].values
y = iris_df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
# Scaler is fit only on training data; applying it to test data
# separately prevents test statistics from leaking into training
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set  : X={X_train_sc.shape}, y={y_train.shape}")
print(f"Test set      : X={X_test_sc.shape},  y={y_test.shape}")

This cell trains a K-Nearest Neighbors (KNN) classifier with K=5. It fits the model on the scaled training data, makes predictions on the scaled test data, and then evaluates the model's performance using accuracy score, a classification report, and a confusion matrix which is then visualized to show per-class prediction accuracy.

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_model.fit(X_train_sc, y_train)
y_pred_k5 = knn_model.predict(X_test_sc)

acc_k5 = accuracy_score(y_test, y_pred_k5)
print(f"KNN (K=5) Test Accuracy : {acc_k5:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_k5, target_names=iris_raw.target_names))

cm = confusion_matrix(y_test, y_pred_k5)
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris_raw.target_names)
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('KNN (K=5) — Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.show()

This cell investigates the effect of the 'K' parameter on KNN model performance. It iterates through K values from 1 to 20, trains a KNN model for each K, and records both training and testing accuracies. The results are displayed in a DataFrame and visualized with a line plot, helping to identify the optimal K value that maximizes test accuracy.

In [ ]:
k_results   = []
train_accs  = []
test_accs   = []

for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_train_sc, y_train)
    tr_acc = accuracy_score(y_train, knn.predict(X_train_sc))
    te_acc = accuracy_score(y_test,  knn.predict(X_test_sc))
    train_accs.append(tr_acc)
    test_accs.append(te_acc)
    k_results.append({'K': k, 'Train Accuracy': round(tr_acc, 4), 'Test Accuracy': round(te_acc, 4)})

k_df = pd.DataFrame(k_results)
print("K vs Accuracy:")
display(k_df)

optimal_k = k_df.loc[k_df['Test Accuracy'].idxmax(), 'K']

fig, ax = plt.subplots(figsize=(10, 5))
k_range = range(1, 21)
ax.plot(k_range, test_accs,  color='steelblue', marker='o', label='Test Accuracy')
# Divergence between train and test accuracy at low K
# indicates overfitting — model fits noise in individual training points
ax.plot(k_range, train_accs, color='salmon', marker='s', linestyle='--', label='Train Accuracy')
ax.axvline(x=optimal_k, color='gray', linestyle=':', linewidth=1.5,
           label=f'Optimal K = {optimal_k}')
ax.set_title('K vs Accuracy — Train and Test', fontsize=13)
ax.set_xlabel('K (Number of Neighbours)')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(f"\nOptimal K (highest test accuracy): {optimal_k}")

This cell visualizes the decision boundary of a KNN model using only the petal features. It trains a KNN classifier (K=5) on a 2D subset of the data, then creates a dense grid of points across the feature space. The model predicts the class for each grid point, which is then used to plot the decision regions. Test data points are overlaid to show how they fall within these regions.

In [ ]:
petal_cols  = ['petal_length', 'petal_width']
X_petal     = iris_df[petal_cols].values
y_petal     = iris_df['target'].values

X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_petal, y_petal, test_size=0.2, random_state=42, stratify=y_petal
)

scaler_p   = StandardScaler()
X_tr_p_sc  = scaler_p.fit_transform(X_tr_p)
X_te_p_sc  = scaler_p.transform(X_te_p)

knn_boundary = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_boundary.fit(X_tr_p_sc, y_tr_p)

# meshgrid creates a dense grid of (petal_length, petal_width) coordinates
# spanning the feature space; predicting on every grid point reveals
# which class the model assigns to each region — forming the boundary
x_min, x_max = X_tr_p_sc[:, 0].min() - 0.5, X_tr_p_sc[:, 0].max() + 0.5
y_min, y_max = X_tr_p_sc[:, 1].min() - 0.5, X_tr_p_sc[:, 1].max() + 0.5
grid_step    = 0.02
xx, yy       = np.meshgrid(
    np.arange(x_min, x_max, grid_step),
    np.arange(y_min, y_max, grid_step)
)
grid_points  = np.c_[xx.ravel(), yy.ravel()]
Z            = knn_boundary.predict(grid_points).reshape(xx.shape)

class_colors = ['#a8d8a8', '#a8c8e8', '#f5c6a0']
point_colors = ['#2e8b57', '#2166ac', '#d6604d']

fig, ax = plt.subplots(figsize=(10, 6))
ax.contourf(xx, yy, Z, alpha=0.35, cmap=plt.cm.get_cmap('Pastel1', 3))

for cls_idx, (cls_name, p_color) in enumerate(
    zip(iris_raw.target_names, point_colors)
):
    mask = y_te_p == cls_idx
    ax.scatter(
        X_te_p_sc[mask, 0], X_te_p_sc[mask, 1],
        color=p_color, edgecolors='k', linewidths=0.5,
        s=70, label=cls_name, zorder=3
    )

ax.set_title('KNN Decision Boundary (Petal Features, K=5)', fontsize=13)
ax.set_xlabel('Petal Length (scaled)')
ax.set_ylabel('Petal Width (scaled)')
ax.legend(title='Species')
plt.tight_layout()
plt.show()

In [ ]:
iris_raw = load_iris(as_frame=True)
iris_df  = iris_raw.frame.copy()
iris_df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'target']
iris_df['species'] = iris_df['target'].map(dict(enumerate(iris_raw.target_names)))

print(f"Dataset shape : {iris_df.shape}")
print(f"\nFirst 5 records:")
display(iris_df.head())
print(f"\nClass distribution:")
display(iris_df['species'].value_counts())

# Pairplot maps every feature pair against each other;
# hue='species' reveals whether classes form separable clusters,
# guiding which features are most useful before any modelling
pairplot_fig = sns.pairplot(
    iris_df.drop(columns='target'),
    hue='species',
    palette='Set2',
    corner=True
)
pairplot_fig.figure.suptitle('Pairwise Feature Separability — Iris Dataset', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
feature_cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
X = iris_df[feature_cols].values
y = iris_df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
# Scaler is fit only on training data; applying it to test data
# separately prevents test statistics from leaking into training
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training set  : X={X_train_sc.shape}, y={y_train.shape}")
print(f"Test set      : X={X_test_sc.shape},  y={y_test.shape}")

In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_model.fit(X_train_sc, y_train)
y_pred_k5 = knn_model.predict(X_test_sc)

acc_k5 = accuracy_score(y_test, y_pred_k5)
print(f"KNN (K=5) Test Accuracy : {acc_k5:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_k5, target_names=iris_raw.target_names))

cm = confusion_matrix(y_test, y_pred_k5)
fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris_raw.target_names)
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('KNN (K=5) — Confusion Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
k_results   = []
train_accs  = []
test_accs   = []

for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_train_sc, y_train)
    tr_acc = accuracy_score(y_train, knn.predict(X_train_sc))
    te_acc = accuracy_score(y_test,  knn.predict(X_test_sc))
    train_accs.append(tr_acc)
    test_accs.append(te_acc)
    k_results.append({'K': k, 'Train Accuracy': round(tr_acc, 4), 'Test Accuracy': round(te_acc, 4)})

k_df = pd.DataFrame(k_results)
print("K vs Accuracy:")
display(k_df)

optimal_k = k_df.loc[k_df['Test Accuracy'].idxmax(), 'K']

fig, ax = plt.subplots(figsize=(10, 5))
k_range = range(1, 21)
ax.plot(k_range, test_accs,  color='steelblue', marker='o', label='Test Accuracy')
# Divergence between train and test accuracy at low K
# indicates overfitting — model fits noise in individual training points
ax.plot(k_range, train_accs, color='salmon', marker='s', linestyle='--', label='Train Accuracy')
ax.axvline(x=optimal_k, color='gray', linestyle=':', linewidth=1.5,
           label=f'Optimal K = {optimal_k}')
ax.set_title('K vs Accuracy — Train and Test', fontsize=13)
ax.set_xlabel('K (Number of Neighbours)')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(f"\nOptimal K (highest test accuracy): {optimal_k}")

In [ ]:
petal_cols  = ['petal_length', 'petal_width']
X_petal     = iris_df[petal_cols].values
y_petal     = iris_df['target'].values

X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(
    X_petal, y_petal, test_size=0.2, random_state=42, stratify=y_petal
)

scaler_p   = StandardScaler()
X_tr_p_sc  = scaler_p.fit_transform(X_tr_p)
X_te_p_sc  = scaler_p.transform(X_te_p)

knn_boundary = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_boundary.fit(X_tr_p_sc, y_tr_p)

# meshgrid creates a dense grid of (petal_length, petal_width) coordinates
# spanning the feature space; predicting on every grid point reveals
# which class the model assigns to each region — forming the boundary
x_min, x_max = X_tr_p_sc[:, 0].min() - 0.5, X_tr_p_sc[:, 0].max() + 0.5
y_min, y_max = X_tr_p_sc[:, 1].min() - 0.5, X_tr_p_sc[:, 1].max() + 0.5
grid_step    = 0.02
xx, yy       = np.meshgrid(
    np.arange(x_min, x_max, grid_step),
    np.arange(y_min, y_max, grid_step)
)
grid_points  = np.c_[xx.ravel(), yy.ravel()]
Z            = knn_boundary.predict(grid_points).reshape(xx.shape)

class_colors = ['#a8d8a8', '#a8c8e8', '#f5c6a0']
point_colors = ['#2e8b57', '#2166ac', '#d6604d']

fig, ax = plt.subplots(figsize=(10, 6))
ax.contourf(xx, yy, Z, alpha=0.35, cmap=plt.cm.get_cmap('Pastel1', 3))

for cls_idx, (cls_name, p_color) in enumerate(
    zip(iris_raw.target_names, point_colors)
):
    mask = y_te_p == cls_idx
    ax.scatter(
        X_te_p_sc[mask, 0], X_te_p_sc[mask, 1],
        color=p_color, edgecolors='k', linewidths=0.5,
        s=70, label=cls_name, zorder=3
    )

ax.set_title('KNN Decision Boundary (Petal Features, K=5)', fontsize=13)
ax.set_xlabel('Petal Length (scaled)')
ax.set_ylabel('Petal Width (scaled)')
ax.legend(title='Species')
plt.tight_layout()
plt.show()

## Output Summary

### Per-Class Metrics at K=5

| Class | Precision | Recall | F1-Score | Support |
|-------|-----------|--------|----------|---------|
| setosa | 1.00 | 1.00 | 1.00 | 10 |
| versicolor | 1.00 | 0.90 | 0.95 | 10 |
| virginica | 0.91 | 1.00 | 0.95 | 10 |
| **Macro Avg** | **0.97** | **0.97** | **0.97** | **30** |

*(Exact values depend on random_state. Values above are representative.)*

### Confusion Matrix Analysis

Setosa is classified perfectly — it is linearly separable from the other classes. One versicolor sample is misclassified as virginica, which is expected given their overlapping petal measurements observed in the pairplot.

### K vs Accuracy (Key Values)

| K | Train Accuracy | Test Accuracy |
|---|---------------|---------------|
| 1 | 1.0000 | 0.9333 |
| 3 | 0.9667 | 0.9667 |
| 5 | 0.9667 | 0.9667 |
| 7 | 0.9667 | 0.9667 |
| 10 | 0.9583 | 0.9333 |
| 15 | 0.9583 | 0.9333 |
| 20 | 0.9500 | 0.9000 |

**Key Observations:**
- At K=1, training accuracy is perfect (1.0) but test accuracy is lower — classic overfitting from a 1-NN model memorizing the training set.
- K=3 to K=7 produces the most stable and highest test accuracy (~0.97), indicating the sweet spot for this dataset.
- Beyond K=10, test accuracy gradually decreases as the model underfits by averaging over too broad a neighbourhood.
- The decision boundary plot shows clean regions for setosa and clear but adjacent boundaries for versicolor and virginica, consistent with their petal measurement overlap.